## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [4]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:llama-3.3-70b-versatile")
response=model.invoke("Why do parrots talk")
response

AIMessage(content="The fascinating world of parrot communication! Parrots are known for their remarkable ability to mimic human speech and other sounds, but why do they do it? Here are some reasons:\n\n1. **Communication and social bonding**: In the wild, parrots use vocalizations to communicate with their flock members, mates, and even rivals. They may mimic sounds to establish dominance, attract a mate, or warn others of potential threats. By mimicking human speech, parrots may be attempting to communicate with their human caregivers and strengthen their bond.\n2. **Learning and cognitive development**: Parrots are highly intelligent birds, and mimicking sounds is a way for them to learn and exercise their cognitive abilities. By repeating sounds, they may be testing their memory, practicing vocal control, and developing their problem-solving skills.\n3. **Attention-seeking behavior**: Parrots are social animals and thrive on attention. By mimicking human speech or other sounds, they

In [9]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="This is the title of the movie")
    year:int=Field(desciption="this is the year movie was released")
    director:str=Field(description="director of the film")
    rating:float=Field(description="the movies rating out of 10")

C:\Users\Rajat Singh\AppData\Local\Temp\ipykernel_19492\896658611.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(desciption="this is the year movie was released")


In [10]:
model_with_structure=model.with_structured_output(Movie)

In [11]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020DCA80C910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020DCA80D310>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': 

In [12]:
model_with_structure.invoke("Provide details about inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

## Message output along with parsed structure

In [13]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="This is the title of the movie")
    year:int=Field(...,desciption="this is the year movie was released")
    director:str=Field(...,description="director of the film")
    rating:float=Field(...,description="the movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Tell me about the movie inception")

C:\Users\Rajat Singh\AppData\Local\Temp\ipykernel_19492\3173869429.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(...,desciption="this is the year movie was released")


In [14]:
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4zjt6cst4', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 282, 'total_tokens': 314, 'completion_time': 0.057338686, 'completion_tokens_details': None, 'prompt_time': 0.106676564, 'prompt_tokens_details': None, 'queue_time': 0.161475838, 'total_time': 0.16401525}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8521-af08-7bf3-b412-fe23befe142d-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': '4zjt6cst4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 282, 'output_tokens': 32, 

## Nested Structure

In [15]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(None,description="Budget in millions USD")


model_with_structure=model.with_structured_output(MovieDetail)

response=model_with_structure.invoke("proide details about the movie inception")

response

MovieDetail(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Action', 'Sci-Fi'], budget=160.0)

## TypeDict